# chat

> The agent the application holds.

One object over: a routing policy, one backend per distinct model, the tool list, the
skills, the extensions, the approval queue and the compactor. Everything above it -- both
frontends, the prompt cell, the CLI -- talks to this and to nothing else in the package.

Two things are done here rather than in a backend callback, and both for the same reason:

**Recording is done by wrapping the tools, not by hooking an engine.** Every tool goes
through `_record`, which logs the call and, for the ones that write, snapshots the file
first. That snapshot is what makes `changes()` report *the file* rather than the model's
account of the file -- a tool result is a claim, and a tool that reported success and
changed nothing simply does not appear. Wrapping works identically on both engines, where
a callback would have to be written twice against two different tool-event shapes.

**Compaction happens between turns, not inside one.** It needs a second model call on a
different model and then has to replace the first model's history; neither engine's
callback system is shaped for that, and doing it out here is what makes the summary
available to write into the notebook.

Everything is lazy and everything is allowed to be missing. Skills are discovered on first
use, extensions on first use, and a backend on the first turn that needs it -- so an
`Agent` costs nothing until something asks it a question, exactly as `Assistant` always
has.


In [ ]:
#| default_exp chat

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import functools, threading
from ramabana.core import agent_err
from ramabana.activity import Activity
from ramabana.backend import Usage, make_backend
from ramabana.compact import Compactor, notices_block
from ramabana.models import Routing, model_note
from ramabana.tools import MAX_TOOL_CHARS, WRITE_TOOLS, tools_for, clip

In [ ]:
#| export
# Skills whose full text goes into the briefing rather than behind `read_skill`.
# exhash earns it: its command grammar is needed by the *first* edit of a session, and a
# model that guesses at it produces commands that fail to parse. Everything else is one
# tool call away, which is where the plan says skills belong.
INLINE_SKILLS = ('exhash',)

In [ ]:
#| export
def system_prompt(host, skills=(), inline=INLINE_SKILLS, extra=''):
    """The agent's briefing: what it is, where it is, how to work, and what it knows.

    The skill index is names and descriptions only -- bodies are a `read_skill` away --
    because a dozen full skill texts would crowd out the code the model is meant to be
    looking at. The exception is spelled out in `INLINE_SKILLS` and is deliberately short.
    """
    from ramabana.skills import find, skill_index
    roots = '\n'.join(f'  {r}' for r in host.roots) or '  (no folder open)'
    # Measured, not assumed. leela serves its inspector from inside the kernel process, so
    # an inspection never joins the shell queue a running cell is holding -- true under
    # ipykernel and ipymini alike (tests/test_subagent_concurrency.py). The earlier version
    # of this line claimed the concurrency came from ipymini; it does not, and telling the
    # model to "keep it short because the kernel is busy" was advice for a problem it does
    # not have.
    conc = ('\n  This works while one of the user\'s cells is still running: leela\'s inspector '
            'lives inside the kernel, so it does not wait for the cell.'
            + (' Your kernel also executes each inspection in its own subshell.'
               if getattr(host, 'concurrent', False) else ''))
    sp = f"""You are the assistant inside leela, a Python IDE. You are working in these folders:
{roots}

You can search the code index (the repo *and* installed packages), read the web, edit
files, run Python in the user's live kernel namespace, and read skills that describe the
tools already installed here.

How to work:
- Before writing code that uses a library, `search_code` for it. The index covers
  installed packages, so prior art usually already exists in the environment.
- Prefer `view_file` over guessing. Its output doubles as the address book for edits.
- To change a file: `view_file` it, then `edit_file` with addresses from that view.
  Work bottom-to-top when making several structural edits in one call.
- `run_python` shares the user's kernel namespace. Read anything; bind results to NEW
  names. You cannot rebind or delete the user's variables, so do not try.
- To *look at* live state, prefer `inspect_python`: neither of its scopes can change what
  the user made, so it needs no approval. Start with the default sandbox and pass
  `scope='overlay'` when it refuses a library call you need.{conc}
- Use `web_search`/`read_url` when the answer depends on current documentation.
- When two or more questions are independent and each would take several tool calls,
  send them together with `delegate_parallel` rather than working through them yourself.
- Writes may be put to the user for approval. A refusal comes back with their reason —
  read it and change the approach, do not retry the same edit.
- Paths must be inside the folders above. Anything else is refused."""
    idx = skill_index(skills)
    if idx: sp += idx
    for name in inline or ():
        if (s := find(skills, name)): sp += f'\n\n## {s.name}\n\n{s.text()}'
    return sp + (f'\n\n{extra}' if extra else '')

In [ ]:
#| export
class Agent:
    """The IDE's agent: a routed chat whose tools are the host's own capabilities.

    `note` and `ready` report availability rather than raising, the way `Search.backend`
    does: the model is a multi-gigabyte download on one side and an API key on the other,
    and an editor that will not open without either is a worse editor.
    """

    def __init__(self,
                 host,
                 model=None,                # the turn model; None takes the routing default
                 routing=None,
                 sp=None,                   # override the whole briefing
                 approvals=None,            # an `Approvals`; None means nothing is gated
                 cfg=None,                  # config dir, for skills and extensions
                 compact=True,              # compact automatically at the threshold
                 kernel_alive=True,         # what the post-compaction note may promise
                 extensions=True,
                 project_extensions=False,  # project extensions execute repo code: opt in
                 ext_paths=(),
                 inline_skills=INLINE_SKILLS,
                 subagents=True,
                 tool_max_len=MAX_TOOL_CHARS,
                 on_compact=None,
                 on_activity=None):         # called as each tool call starts and finishes
        self.host, self.cfg, self.inline_skills = host, cfg, inline_skills
        self.routing = routing or Routing(turn=model)
        if model: self.routing.set(model)
        self.approvals, self.tool_max_len, self.subagents = approvals, tool_max_len, subagents
        self.extensions, self.project_extensions, self.ext_paths = extensions, project_extensions, ext_paths
        self._sp = sp
        self.compactor = Compactor(auto=compact, kernel_alive=kernel_alive, on_compact=on_compact)
        self.activity = Activity(on_change=on_activity)   # the live account of what it is doing
        self.calls = []          # (tool, args) per call this session -- what the UI shows as activity
        self.before = {}         # path -> its text just before a write tool touched it, this turn
        self.use = Usage()       # this session's total, across every model it routed to
        self.note = 'not started'
        self._backends, self._skills, self._reg, self._tools = {}, None, None, None
        self.lock = threading.Lock()

    # -- what it knows -------------------------------------------------------
    @property
    def skills(self):
        "Every discovered skill, found once. Includes anything an extension registered."
        if self._skills is None:
            from ramabana.skills import discover
            try: self._skills = discover(self.host.roots, self.cfg, extra=self.registry.skills)
            except Exception as e:
                self._skills = []
                self.note = f'skills unavailable ({agent_err(e)})'
        return self._skills

    @property
    def registry(self):
        "The extension registry, loaded once. Empty when extensions are switched off."
        if self._reg is None:
            from ramabana.extensions import Registry, load
            self._reg = Registry(host=self.host, agent=self)
            if self.extensions:
                try: load(self._reg, self.host.roots, self.cfg, self.project_extensions, self.ext_paths)
                except Exception as e: self._reg.notes.append(f'extension loading failed: {agent_err(e)}')
        return self._reg

    @property
    def tools(self):
        "Every tool, built once and recorded. Rebuilt by `reload`."
        if self._tools is None:
            extra = list(self.registry.tools)
            if self.subagents:
                from ramabana.subagent import subagent_tools
                extra += subagent_tools(lambda: self._be_or_none('subagent'), lambda: self._plain)
            plain = tools_for(self.host, lambda: self.skills, extra)
            self._plain = plain
            self._tools = [self._record(t) for t in plain]
        return self._tools

    def reload(self):
        "Re-discover skills, extensions and tools. What a `/reload` command calls after editing them."
        self._skills = self._reg = self._tools = None
        for b in self._backends.values(): b.close()
        self._backends.clear()
        return self

    def system_prompt(self):
        return self._sp or system_prompt(self.host, self.skills, self.inline_skills)

    # -- recording -----------------------------------------------------------
    def _record(self, f):
        """Wrap one tool so its call is logged and its damage is measurable.

        `functools.wraps` is load-bearing rather than tidy: both backends build their tool
        schema from the function's signature, docstring and annotations, and `__wrapped__`
        is what lets `inspect` see through to the real one. Without it every tool would be
        described to the model as `(*args, **kw)` with no documentation.

        The snapshot has to happen here because this is the last moment the `before` still
        exists. First touch only -- later edits to the same file in one turn are part of
        one change.
        """
        name = getattr(f, '__name__', '?')

        @functools.wraps(f)
        def wrapper(*a, **kw):
            args = kw if kw else _named(f, a)
            self.calls.append((name, args))
            act = self.activity.start(name, args)
            self.registry.fire('before_tool', self, name, args)
            if name in WRITE_TOOLS and (p := args.get('path')) and p not in self.before:
                self.before[p] = self.host.text_at(p) or ''
            try:
                out = f(*a, **kw)
            except Exception as e:
                self.activity.finish(act, agent_err(e), ok=False)
                raise
            self.activity.finish(act, out, ok=not str(out).startswith(('edit failed', 'write failed', 'could not')))
            self.registry.fire('after_tool', self, name, out)
            return out
        return wrapper

    def changes(self):
        """`{path: (before, after)}` for every file this turn's write tools actually moved.

        A tool that reported success but changed nothing does not appear here, which is the
        whole point: this is the file, not the claim about the file.
        """
        out = {}
        for p, was in self.before.items():
            now = self.host.text_at(p)
            if now is not None and now != was: out[p] = (was, now)
        return out

    # -- backends ------------------------------------------------------------
    def _be(self, job='turn'):
        """The backend for `job`, built on first use and shared by every job on the same model.

        Shared by model rather than by job on purpose. If summaries and turns happen to
        resolve to the same local model, building two backends would load the engine twice
        -- gigabytes, for no reason. The tools go on whichever backend is the turn model's;
        every other job reaches the engine through `oneshot` or `spawn`, neither of which
        wants them.
        """
        spec = self.routing.spec(job)
        key = (spec.backend, spec.model_id)
        if key not in self._backends:
            is_turn = key == (lambda s: (s.backend, s.model_id))(self.routing.spec('turn'))
            kw = {}
            if is_turn:
                kw = dict(sp=self.system_prompt(), tools=self.tools, tool_max_len=self.tool_max_len,
                          approve=(self.approvals.gate if self.approvals is not None else None))
            self._backends[key] = make_backend(spec, **kw)
        return self._backends[key]

    def _be_or_none(self, job='turn'):
        "The backend for `job` if it can start, else None. What a tool asks, since a tool cannot raise usefully."
        try:
            b = self._be(job)
            return b if b.start() is not None else None
        except Exception: return None

    @property
    def backend(self): return self._be('turn')

    @property
    def chat(self):
        "The live chat object, or None. Kept for the frontends, which use it to cancel."
        return self._be('turn').chat

    @property
    def ready(self):
        """Whether the turn model is up.

        Asked of the backend rather than looked up in the cache. Building a `Backend` is
        just an object -- the expensive part is `start()`, which this does not call -- and
        reaching into `_backends` by key meant anything that supplied a backend some other
        way (a test, a sub-agent harness) reported itself as permanently not ready.
        """
        return self._be('turn').ready

    @property
    def busy(self): return self.lock.locked()

    @property
    def model(self): return self.routing.spec('turn')

    def start(self):
        "Build the turn backend, once. Returns it, or None with `note` explaining why not."
        b = self._be('turn')
        if b.start() is None:
            self.note = b.note
            return None
        # From the backend that is actually running, not from the routing table: a status
        # line that names a model the turn is not on is worse than no status line.
        self.note = f'{model_note(b.spec)} · {len(self.tools)} tools'
        return b

    def retry(self):
        "Forget a previous failure, so a model that has since downloaded or been keyed is picked up."
        b = self._be('turn')
        b.retry()
        return self.start()

    def set_model(self, name, job='turn'):
        "Point `job` at `name`. The old backend is closed, because two loaded local models is one too many."
        old = {(s.backend, s.model_id) for s in (self.routing.spec(j) for j in ('turn',))}
        spec = self.routing.set(name, job)
        for k in list(self._backends):
            if k in old and k != (spec.backend, spec.model_id):
                self._backends.pop(k).close()
        self.note = f'{job} → {model_note(spec)}'
        return spec

    # -- turns ---------------------------------------------------------------
    def _prepare(self, prompt):
        "Everything that happens before a message goes out: notices, hooks, compaction."
        self.before.clear()                    # `changes()` reports this turn, not the session
        self.activity.mark()                   # and so does `turn_md()`
        self.registry.fire('before_turn', self, prompt)
        if self.compactor.auto and self.compactor.due(self._be('turn')): self.compact()
        return prompt

    def _finish(self, text):
        b = self._be('turn')
        self.use = self.use + b.use
        self.registry.fire('after_turn', self, text)
        return text

    def ask(self, prompt, **kw):
        "One turn. Returns the assistant's text, or the reason there isn't any."
        if self.start() is None: return self.note
        with self.lock:
            try:
                self._prepare(prompt)
                return self._finish(self._be('turn').send(_with_notices(prompt), **kw))
            except Exception as e:
                self.note = f'the assistant failed ({agent_err(e)})'
                return self.note

    def stream(self, prompt, **kw):
        "One turn as an iterator of markdown chunks, for a frontend that can render as it arrives."
        if self.start() is None:
            yield self.note
            return
        with self.lock:
            try:
                self._prepare(prompt)
                out = []
                for chunk in self._be('turn').stream(_with_notices(prompt), **kw):
                    out.append(chunk)
                    yield chunk
                self._finish(''.join(out))
            except Exception as e:
                self.note = f'the assistant failed ({agent_err(e)})'
                yield f'\n\n{self.note}'

    def compose(self, prompt, context='', screen='', image=None):
        """One message from whatever the frontend can supply: the notebook above the
        question, a captured screen as text, and a captured screen as a picture.

        `image` goes in as a *content part* rather than as a tool result, which is both the
        supported path to a multimodal turn on either backend and the right place for it --
        alongside the question asked about it, instead of behind a tool the model has to
        think to call.

        `screen` is the terminal frontend's equivalent and stays text on purpose: a
        terminal screen is a grid of characters, so sending it as prose is both exact and
        far cheaper than a rendering of it would be.

        Separate from `ask_with` so `stream_with` composes identically. A streamed turn
        that quietly saw a different message from a blocking one would be a very hard bug
        to find.
        """
        parts = []
        if context: parts.append(f'<notebook>\n{context}\n</notebook>')
        if screen: parts.append(f'<screen>\n{screen}\n</screen>')
        parts.append(prompt)
        ask = '\n\n'.join(parts)
        return [image, ask] if image else ask

    def ask_with(self, prompt, context='', screen='', image=None):
        "One turn with the frontend's context attached. Blocking; see `stream_with` for the live one."
        return self.ask(self.compose(prompt, context, screen, image))

    def stream_with(self, prompt, context='', screen='', image=None):
        "The same turn, as an iterator of markdown chunks."
        return self.stream(self.compose(prompt, context, screen, image))

    def cancel(self):
        "Stop the turn in flight, and release anyone waiting on an approval for it."
        if self.approvals is not None: self.approvals.cancel_all('the turn was stopped')
        return bool(self._be('turn').cancel())

    def close(self):
        for b in list(self._backends.values()):
            try: b.close()
            except Exception: pass
        self._backends.clear()

    # -- the cheap jobs ------------------------------------------------------
    def oneshot(self, prompt, sp='', job='classify', max_tokens=None):
        "A question on whichever model `job` routes to, in a conversation that is thrown away."
        b = self._be_or_none(job)
        return '' if b is None else b.oneshot(prompt, sp, max_tokens)

    def classify(self, text, labels):
        "One label for `text`, on the cheap model. Returns the matched label, or the raw reply."
        out = self.oneshot(f'{text}\n\nChoose exactly one label from: {", ".join(labels)}.',
                           'Reply with only the single best label and nothing else.', 'classify', 32).lower()
        return next((l for l in labels if l.lower() in out), out.strip())

    def summarise(self, text, sp='Summarise concisely. Output only the summary.'):
        return self.oneshot(text, sp, 'summary')

    def compact(self, extra=''):
        """Compact the conversation now. Returns the summary text, or `''` with `compactor.note` set.

        The summarizing runs on the `summary` model -- local by default. Paying frontier
        prices to compress a frontier conversation is exactly the spending routing exists
        to stop, and the job is a mechanical transformation of a transcript we already hold.
        """
        b = self._be('turn')
        if b.chat is None:
            self.compactor.note = 'nothing to compact: the model is not running'
            return ''
        sub = self._be_or_none('summary')
        summariser = (sub.oneshot if sub is not None else b.oneshot)
        text = self.compactor.compact(b, lambda p, sp: summariser(p, sp, 4000), extra)
        if text: self.registry.fire('compact', self, text)
        self.note = self.compactor.note
        return text

    # -- what to show --------------------------------------------------------
    @property
    def pct_full(self):
        try: return self._be('turn').pct_full
        except Exception: return 0.0

    def turn_md(self, title='what I did'):
        "This turn's tool calls as foldable markdown, to save alongside the answer in a cell."
        return self.activity.md(mark=self.activity._mark, title=title)

    def turn_lines(self):
        "This turn's tool calls as plain summary lines, for a pane that cannot fold."
        return self.activity.lines(mark=self.activity._mark)

    @property
    def problems(self):
        """Everything that went wrong and had nowhere to be reported, newest last.

        Gathered from every backend rather than kept here, because most of these happen on
        the *cheap* model -- a compaction that could not run, a completion the engine
        refused, an engine complaining on stderr in native code where nothing can catch it.
        Each of those returns `''` to a caller that cannot raise, and until this existed,
        `''` was the whole story the user got.
        """
        out = []
        for b in self._backends.values():
            for p in b.problems:
                if p not in out: out.append(p)
        if (n := self.compactor.note).startswith('compaction') and n not in out: out.append(n)
        return out[-10:]

    def clear_problems(self):
        "Forget them, for a frontend that has shown them."
        for b in self._backends.values(): b.problems.clear()
        return self

    def status(self):
        "Everything a status bar or an `/agent` command wants, in one dict."
        return {'ready': self.ready, 'busy': self.busy, 'note': self.note,
                'problems': self.problems,
                'model': self.model.name, 'model_note': model_note(self.model),
                'ntools': len(self.tools), 'nskills': len(self.skills),
                'pct_full': round(self.pct_full, 3), 'compactions': self.compactor.count,
                'use': self.use.dict(), 'usage': repr(self.use),
                'activity': self.activity.rows(40),
                'approval': (self.approvals.pending.dict() if self.approvals is not None
                             and self.approvals.pending is not None else None),
                'calls': [{'tool': t, 'args': str(a)[:300]} for t, a in self.calls[-40:]]}

    def command(self, line):
        """Run a slash command. Returns text to show, or None when the command is unknown.

        Here rather than in the frontends because both of them need every one of these, and
        a command that exists in the terminal and not the browser is the kind of drift
        `keys.py` was written to prevent.
        """
        line = (line or '').strip().lstrip('/')
        name, _, arg = line.partition(' ')
        arg = arg.strip()
        if name == 'model':
            if not arg: return self.routing.summary()
            job, _, m = arg.partition(' ')
            try: return f'{model_note(self.set_model(m or job, job if m else "turn"))}'
            except Exception as e: return agent_err(e)
        if name == 'cost': return repr(self.use)
        if name == 'compact':
            t = self.compact(arg)
            return f'{self.compactor.note}\n\n{t}' if t else self.compactor.note
        if name == 'skills':
            return '\n'.join(f'{s.name:16} {s.source:8} {s.description[:90]}' for s in self.skills) or 'no skills found'
        if name == 'skill': return self.oneshot and clip(_skill_text(self.skills, arg))
        if name == 'tools': return '\n'.join(sorted(getattr(t, '__name__', '?') for t in self.tools))
        if name == 'extensions': return '\n'.join(self.registry.notes) or 'no extensions loaded'
        if name == 'reload':
            self.reload()
            return f'reloaded: {len(self.tools)} tools, {len(self.skills)} skills'
        if name in self.registry.commands:
            fn, _ = self.registry.commands[name]
            try: return fn(self, arg)
            except Exception as e: return agent_err(e)
        return None

    def commands(self):
        "Every command name, built-in and registered, for a help line or an autocomplete."
        return sorted({'model', 'cost', 'compact', 'skills', 'skill', 'tools', 'extensions', 'reload',
                       *self.registry.commands})

In [ ]:
#| export
def _named(f, a):
    """Positional arguments as a dict, so an activity line can name them.

    Models normally send named arguments, but an extension's tool may be called
    positionally by another extension, and an activity line reading `edit_file(...)` with
    no path is exactly the line that makes the feed useless.
    """
    if not a: return {}
    try:
        import inspect
        return dict(zip(list(inspect.signature(f).parameters), a))
    except Exception: return {'args': a}

In [ ]:
#| export
def _skill_text(skills, name):
    from ramabana.skills import find
    s = find(skills, name)
    return f'no skill matching {name!r}' if s is None else s.text()

In [ ]:
#| export
def _with_notices(prompt):
    "Append the aai-coding style prompt notices, when the prompt earns any. Text prompts only."
    if not isinstance(prompt, str): return prompt
    return prompt + notices_block(prompt)

In [ ]:
#| export
# ---------------------------------------------------------------------------
# inline completion
# ---------------------------------------------------------------------------
COMPLETE_SP = """You are a code completion engine inside an editor. You are given the code before \
the cursor in <before> and the code after it in <after>.

Reply with ONLY the code that belongs at the cursor. No explanation, no markdown fence, no \
repetition of <before> or <after>. Keep it short -- finish the current expression, statement or \
short block and stop. Match the surrounding indentation and style exactly. If nothing sensible \
belongs there, reply with nothing at all."""

In [ ]:
#| export
MAX_COMPLETION_LINES = 4     # a suggestion longer than this is a guess about the design, not a completion

In [ ]:
#| export
COMPLETION_TOKENS = 96

In [ ]:
#| export
CTX_BEFORE, CTX_AFTER = 2000, 600   # chars of surrounding code sent as context

In [ ]:
#| export
def _strip_echo(before, out):
    "Drop a re-emitted tail of `before` from the front of `out` -- models like to restate the line they continue."
    tail = before[-200:]
    for n in range(len(tail), 0, -1):
        if out.startswith(tail[-n:]): return out[n:]
    return out

In [ ]:
#| export
def _clean(text, before, max_lines):
    """A raw model reply as something safe to insert: fences off, echo off, `max_lines` long.

    `fenced_blocks` rather than a ```python-only matcher because a completion model asked
    for bare code and fencing it anyway rarely bothers with an info string. It also keeps
    this function testable without a model installed.
    """
    from fastcore.xtras import fenced_blocks
    out = (blocks[-1][1] if (blocks := fenced_blocks(text or '')) else text) or ''
    out = _strip_echo(before, out.strip('\n'))
    lines = out.split('\n')[:max_lines]
    while lines and not lines[-1].strip(): lines.pop()
    return '\n'.join(lines)

In [ ]:
#| export
class Completer:
    """Inline completion, always on the local model.

    Routing sends `completion` to the local backend unconditionally, and that is the whole
    argument for routing in one feature: a completion is four lines of code, fires
    constantly, has to feel instant, and is worth approximately nothing per call. Sending
    it to a frontier model would be slower, cost real money, and put every keystroke's
    surroundings on someone else's wire.

    A completion is a *question about the code on screen*, not a turn in a conversation, so
    it runs in a throwaway conversation on an engine that is already loaded. Letting
    suggestions accumulate would poison the assistant's context as well as their own.

    Nothing here is automatic. The kernel's own completions are instant and correct and
    stay bound to typing; this costs a forward pass, so it fires only when asked for.
    """

    def __init__(self, agent, max_lines=MAX_COMPLETION_LINES, max_tokens=COMPLETION_TOKENS):
        self.a, self.max_lines, self.max_tokens = agent, max_lines, max_tokens
        self.note = 'not asked yet'

    @property
    def ready(self):
        b = self.a._be_or_none('completion')
        return b is not None

    def _prompt(self, code, pos, lang):
        return (f'Language: {lang}\n\n<before>\n{code[:pos][-CTX_BEFORE:]}\n</before>\n'
                f'<after>\n{code[pos:][:CTX_AFTER]}\n</after>')

    def complete(self, code, pos, lang='python'):
        "The text to insert at `pos` in `code`, or `''` with `note` saying why there isn't any."
        b = self.a._be_or_none('completion')
        if b is None:
            self.note = 'no completion model available'
            return ''
        if b.busy:
            # One engine means one generation at a time. Declining beats queueing behind a
            # tool loop the user is watching: "the model is thinking" is a better answer
            # than an editor that has stopped taking keys.
            self.note = 'model busy — it is mid-turn'
            return ''
        text = b.oneshot(self._prompt(code, pos, lang), COMPLETE_SP, self.max_tokens)
        if not text:
            self.note = b.note if not b.ready else 'no suggestion'
            return ''
        out = _clean(text, code[:pos], self.max_lines)
        self.note = f'{len(out.splitlines())} line(s) from {b.spec.name}' if out else 'no suggestion'
        return out

## Tests


In [ ]:
# A whole turn, driven with no model: the facade the application holds, over a scripted
# backend. This is the shape `leela/ai.py` binds a workspace to.
from ramabana.testing import MemHost, fake_agent
a, be = fake_agent(MemHost({'/proj/a.py': 'x = 1\n'}), replies=['I read the file.'])
print(a.ask('what is in a.py?'))
print('usage :', a.use)
print('model :', a.model.name)

In [ ]:
# The recorder has to be transparent to `inspect`, because both backends build their tool
# schema from the signature and the docstring. Wrap it wrong and every tool arrives as
# `(*args, **kw)` and the model cannot call anything.
import inspect
t = next(t for t in a.tools if t.__name__ == 'view_file')
print('signature:', inspect.signature(t))
print('doc      :', (t.__doc__ or '').splitlines()[0])
assert list(inspect.signature(t).parameters) == ['path', 'start', 'end']
assert t.__doc__

In [ ]:
# `changes()` reports what is different on disk, not what a tool claimed. A tool that said
# "done" and wrote nothing must not show up in the diff.
host = MemHost({'/proj/a.py': 'x = 1\n'})
a2, _ = fake_agent(host, replies=['done'])
a2.ask('change nothing')
print('changes after a no-op turn:', a2.changes())
assert not a2.changes()

In [ ]:
# The system prompt is built from what the host can actually do, so a host with no kernel
# does not get an agent that keeps promising to run code.
sp = a.system_prompt()
print(sp[:700])
assert '/proj' in sp            # it is told which folders it is confined to
assert len(sp) > 200